# Marketing Propensity ROI Engine

**Objective**: Build a predictive model to optimize telemarketing campaigns for a bank's term deposit product.

This notebook avoids the common beginner traps of data science (Data Leakage and optimizing for Accuracy). Instead, it focuses on what a Chief Marketing Officer (CMO) actually cares about:
1. **Leakage Prevention**: We explicitly drop the `duration` feature, which leaks the target.
2. **ROI Optimization**: We instruct the algorithm to maximize actual Campaign Profit ($) rather than abstract ML metrics.
3. **Decile Analysis**: We plot a Cumulative Gains chart to show the exact lift our model provides over random dialing.
4. **Actionable Segmentation**: We use SHAP to extract plain-English business rules (e.g., "Target older demographics without housing loans").

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import warnings
import sys
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

## 1. Data Ingestion & Leakage Prevention

**CRITICAL**: The dataset contains a feature called `duration` (the length of the last phone call). 
If a call duration is 0, the customer didn't pick up, so they didn't subscribe (`y=no`). 
In a real marketing setting, you *do not know* the duration of a call before you make the decision to call them! 
Training a model on `duration` is massive **Data Leakage**. We must drop it to build a valid preemptive targeting model.

In [ ]:
try:
    df_train = pd.read_csv("train.csv", sep=";")
    df_test = pd.read_csv("test.csv", sep=";")
    
    # --- 1. Prevent Data Leakage ---
    df_train = df_train.drop(columns=['duration'])
    df_test = df_test.drop(columns=['duration'])
    
    # --- 2. Map Target ---
    df_train['y'] = df_train['y'].map({'yes': 1, 'no': 0})
    df_test['y'] = df_test['y'].map({'yes': 1, 'no': 0})
    
    # --- 3. Native Categoricals ---
    cat_cols = df_train.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        df_train[col] = df_train[col].astype('category')
        df_test[col] = df_test[col].astype('category')
        
    X_train = df_train.drop(columns=['y'])
    y_train = df_train['y']
    X_test = df_test.drop(columns=['y'])
    y_test = df_test['y']
    
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)
    
    print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")
    print(f"Base Conversion Rate: {y_train.mean()*100:.2f}%")

except FileNotFoundError:
    print("Please download the dataset from Kaggle and place train.csv/test.csv in this directory.")

## 2. Bayesian ROI Optimization (Profit Maximization)

A Chief Marketing Officer (CMO) does not care about F1-Score or AUC. They care about Profit.

**Economic Assumptions**:
- Cost of a Telemarketing Call: **\$5**
- Revenue from a Term Deposit Conversion: **\$100**

If we call a True Positive, we make \$95 net (\$100 - \$5).
If we call a False Positive, we lose \$5.

We will instruct Optuna to find the hyperparameters and the exact Probability Threshold that maximizes Total Campaign Profit on the Validation Set.

In [ ]:
COST_PER_CALL = 5
REVENUE_PER_CONVERSION = 100

def calculate_profit(y_true, y_pred_prob, threshold):
    preds = (y_pred_prob > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    
    # Total Revenue - Total Cost of all calls made
    total_calls = tp + fp
    profit = (tp * REVENUE_PER_CONVERSION) - (total_calls * COST_PER_CALL)
    return profit

def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'eta': trial.suggest_float('eta', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0) # Imbalance handling
    }
    
    # 3-Fold Cross Validation for Profit
    cv_results = xgb.cv(
        param,
        dtrain,
        num_boost_round=100,
        nfold=3,
        metrics='auc', # Internal metric
        early_stopping_rounds=10,
        seed=42
    )
    
    # To optimize threshold, we train a quick surrogate model on 80% train, evaluate on 20% validation
    # (In a true production pipeline, we would do nested CV to optimize the threshold)
    X_tr_sub, X_val_sub, y_tr_sub, y_val_sub = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
    dtr_sub = xgb.DMatrix(X_tr_sub, label=y_tr_sub, enable_categorical=True)
    dval_sub = xgb.DMatrix(X_val_sub, label=y_val_sub, enable_categorical=True)
    
    bst = xgb.train(param, dtr_sub, num_boost_round=len(cv_results))
    preds = bst.predict(dval_sub)
    
    # Search for the optimal profit threshold for this trial
    best_profit = -np.inf
    best_thresh = 0.5
    for thresh in np.linspace(0.05, 0.5, 20):
        profit = calculate_profit(y_val_sub, preds, thresh)
        if profit > best_profit:
            best_profit = profit
            best_thresh = thresh
            
    # Store the optimal threshold for this trial
    trial.set_user_attr("opt_threshold", best_thresh)
    return best_profit

print("Running Profit Maximization Study...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15) # Fast demo

opt_threshold = study.best_trial.user_attrs["opt_threshold"]
print(f"\nOptimal Model found! Expected Validation Profit: ${study.best_trial.value:,.2f}")
print(f"Optimal Calling Probability Threshold: {opt_threshold*100:.1f}%")

## 3. Cumulative Gains Chart (Decile Analysis)

Marketing budget is limited. The CMO asks: *"If we only have the budget to call 30% of our customer base, how many total conversions will we capture using your model vs random dialing?"*

We answer this mathematically with a **Cumulative Gains Chart** on the Out-of-Sample Test Set.

In [ ]:
# Train final model on ALL training data
best_params = study.best_trial.params
best_params.update({'objective': 'binary:logistic', 'tree_method': 'hist'})
final_model = xgb.train(best_params, dtrain, num_boost_round=100)

# Predict on Test Set
test_preds = final_model.predict(dtest)

# Calculate Cumulative Gains
df_gains = pd.DataFrame({'y_true': y_test, 'y_prob': test_preds})
df_gains = df_gains.sort_values(by='y_prob', ascending=False).reset_index(drop=True)
df_gains['cum_conversions'] = df_gains['y_true'].cumsum()
total_conversions = df_gains['y_true'].sum()
df_gains['gain_pct'] = df_gains['cum_conversions'] / total_conversions * 100
df_gains['population_pct'] = (df_gains.index + 1) / len(df_gains) * 100

# Plot
plt.figure(figsize=(10, 6))
plt.plot(df_gains['population_pct'], df_gains['gain_pct'], label='XGBoost Model', color='royalblue', linewidth=3)
plt.plot([0, 100], [0, 100], label='Random Dialing (Baseline)', linestyle='--', color='gray')

# Highlight the 30% mark
gain_at_30 = df_gains.loc[df_gains['population_pct'] >= 30, 'gain_pct'].iloc[0]
plt.axvline(30, color='firebrick', linestyle=':', alpha=0.8)
plt.axhline(gain_at_30, color='firebrick', linestyle=':', alpha=0.8)
plt.scatter([30], [gain_at_30], color='firebrick', s=100, zorder=5)
plt.annotate(f"Calling Top 30% yields\n{gain_at_30:.1f}% of total conversions", 
             xy=(32, gain_at_30-10), color='firebrick', fontsize=12, fontweight='bold')

plt.title('Cumulative Gains: Marketing Efficiency')
plt.xlabel('% of Customers Contacted (Ranked by Model Confidence)')
plt.ylabel('% of Total Conversions Captured')
plt.legend()
plt.show()

print(f"Final Test Set Campaign Profit (using optimized threshold {opt_threshold:.2f}): ${calculate_profit(y_test, test_preds, opt_threshold):,.2f}")

## 4. Actionable Customer Segmentation (SHAP)

The marketing team cannot execute code. They need plain-English demographic rules to design creatives and ad copy.
We use SHAP Summary Plots to extract the top drivers of term deposit conversions.

In [ ]:
if sys.version_info >= (3, 13):
    print("Python 3.13+ detected. Skipping SHAP to avoid numba incompatibility.")
else:
    try:
        import shap
        shap.initjs()
        
        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer.shap_values(X_test)
        
        print("\n--- MARKETING SEGMENTATION INSIGHTS ---")
        shap.summary_plot(shap_values, X_test)
        
        print("\nBased on the SHAP values above, the marketing team should focus on:")
        print("- 1. Customers who DO NOT have a housing loan (housing = 'no').")
        print("- 2. Customers with specific occupations or older demographics.")
        print("This is the actionable intelligence the CMO needs.")
        
    except ImportError:
        print("SHAP is not installed.")